In [ ]:
import tensorflow as tf
from livelossplot import PlotLossesKeras
import matplotlib.pyplot as plt
import numpy as np
from utils.load import load_data
from utils.handlers import get_average
import math
from PIL import Image
import torchvision
import pickle

In [ ]:
tf.__version__

In [ ]:
transform = torchvision.transforms.Compose([
    torchvision.transforms.RandomAffine(degrees=3, scale=(.8, 1.1)),
    torchvision.transforms.RandomPerspective(.3),
    torchvision.transforms.RandomAutocontrast(),
    torchvision.transforms.ColorJitter(.5,.5,.5,.3),
    torchvision.transforms.RandomInvert(.3),

])


In [ ]:
class MySequense(tf.keras.utils.Sequence):
    def __init__(self, x_set, y_set, batch_size):
        self.x, self.y = x_set, y_set
        self.batch_size = batch_size

    def __len__(self):
        return math.ceil(len(self.x) / self.batch_size)

    def __getitem__(self, idx):
        batch_x = self.x[idx * self.batch_size:(idx + 1) *
                                               self.batch_size]
        batch_y = self.y[idx * self.batch_size:(idx + 1) *
                                               self.batch_size]

        return np.array([
            np.asarray(transform(Image.fromarray(np.uint8(x)))) / 255.
            for x in batch_x]), np.array(batch_y)

In [ ]:
def defective_pixels(images_data):
    for im in range(len(images_data)):
        for row in range(1, len(images_data[im]) - 1):
            for px in range(1, len(images_data[im][row]) - 1):
                if np.array_equal(images_data[im][row][px], np.array([1., 1., 1.])) or np.array_equal(
                        images_data[im][row][px], np.array([0., 0., 0.])):
                    images_data[im][row][px] = np.array([np.median([images_data[im][row - 1][px][0],
                                                                    images_data[im][row - 1][px - 1][0],
                                                                    images_data[im][row - 1][px + 1][0],
                                                                    images_data[im][row + 1][px][0],
                                                                    images_data[im][row + 1][px - 1][0],
                                                                    images_data[im][row + 1][px + 1][0],
                                                                    images_data[im][row][px + 1][0],
                                                                    images_data[im][row][px - 1][0]]),
                                                         np.median([images_data[im][row - 1][px][1],
                                                                    images_data[im][row - 1][px - 1][1],
                                                                    images_data[im][row - 1][px + 1][0],
                                                                    images_data[im][row + 1][px][1],
                                                                    images_data[im][row + 1][px - 1][1],
                                                                    images_data[im][row + 1][px + 1][1],
                                                                    images_data[im][row][px + 1][1],
                                                                    images_data[im][row][px - 1][1]]),
                                                         np.median([images_data[im][row - 1][px][2],
                                                                    images_data[im][row - 1][px - 1][2],
                                                                    images_data[im][row - 1][px + 1][2],
                                                                    images_data[im][row + 1][px][2],
                                                                    images_data[im][row + 1][px - 1][2],
                                                                    images_data[im][row + 1][px + 1][2],
                                                                    images_data[im][row][px + 1][2],
                                                                    images_data[im][row][px - 1][2]])])
    return images_data[0:len(images_data), 1:-1, 1:-1, 0:3]

In [ ]:

def load_data(file_name: str) -> np.ndarray:
    """
    :param file_name: path of filename
    :return: read dataset
    """

    with open(file_name, 'rb') as input_file:
        dataframe = pickle.load(input_file)
    return dataframe


In [ ]:
df = load_data('data_train')
labels = df['labels']

In [ ]:
arr = df['images'].astype(np.uint8)


arr = defective_pixels(arr/255.)

In [ ]:
with open('arr.np.ndarray', 'wb') as f:
    pickle.dump(arr, f)

In [ ]:
with open('arr.np.ndarray', 'rb') as f:
    arr = pickle.load(f)

In [ ]:
arr.shape


In [ ]:
plt.figure(figsize=(15,15))
for i in range(9):
    plt.subplot(3,3,i+1)
    plt.xticks([])
    plt.yticks([])
    plt.grid(False)
    plt.imshow(np.array(arr[i]))
    plt.xlabel(labels[i])

plt.show()

In [ ]:
arr = np.uint8(arr * 255)
arr

In [ ]:
from sklearn.model_selection import train_test_split

arr, validation_images, labels, validation_labels = train_test_split(arr, labels, test_size=0.05)

In [ ]:
validation_images = validation_images / 255.

In [ ]:
ds = MySequense(arr, labels, batch_size=256)


In [ ]:

val_dataset = tf.data.Dataset.from_tensor_slices((validation_images,
                                                  validation_labels))

In [ ]:
input_shape = (30, 30, 3)
input_shape

In [ ]:
batch_size = 256

val_dataset = val_dataset.batch(batch_size)

In [ ]:
ds[
    8]

In [ ]:
input_layer = tf.keras.layers.Input(shape=input_shape)
c = tf.keras.layers.CenterCrop(27, 27)(input_layer)
# Первый блок:
b1l1 = tf.keras.layers.Conv2D(32, 3, activation="relu", padding="same")(c)
b1l2 = tf.keras.layers.Conv2D(64, 3, activation="relu", padding="same")(b1l1)
b1_output = tf.keras.layers.MaxPooling2D()(b1l2)

# Второй блок:
b2l1 = tf.keras.layers.Conv2D(64, 3, activation="relu", padding="same")(b1_output)
b2l2 = tf.keras.layers.Conv2D(64, 3, activation="relu", padding="same")(b2l1)

# Связываем эти 2 блока
b2b1_output = tf.keras.layers.add([b2l2, b1_output])

# Третий блок:
b3l1 = tf.keras.layers.Conv2D(64, 3, activation="relu", padding="same")(b2b1_output)
b3l2 = tf.keras.layers.Conv2D(64, 3, activation="relu", padding="same")(b3l1)

# Связываем эти 2 блока
block_3_output = tf.keras.layers.add([b3l2, b2b1_output])

b4l1 = tf.keras.layers.Conv2D(64, 3, activation="relu", padding="same")(block_3_output)
b4l2 = tf.keras.layers.Conv2D(64, 3, activation="relu", padding="same")(b4l1)

block_4_output = tf.keras.layers.add([block_3_output, b4l2])

b5l1 = tf.keras.layers.Conv2D(64, 3, activation="relu", padding="same")(block_4_output)
b5l2 = tf.keras.layers.Conv2D(64, 3, activation="relu", padding="same")(b5l1)

block_5_output = tf.keras.layers.add([block_4_output, b5l2])

b6l1 = tf.keras.layers.Conv2D(64, 3, activation="relu", padding="same")(block_5_output)
b6l2 = tf.keras.layers.Conv2D(64, 3, activation="relu", padding="same")(b6l1)

block_6_output = tf.keras.layers.add([block_5_output, b6l2])

l1 = tf.keras.layers.Conv2D(64, 3, activation="relu", padding="same")(block_6_output)
l2 = tf.keras.layers.GlobalAveragePooling2D()(l1)
l4 = tf.keras.layers.Dense(256, activation="relu")(l2)
bn = tf.keras.layers.BatchNormalization()(l4)
l5 = tf.keras.layers.Dropout(.1)(bn)
outputs = tf.keras.layers.Dense(10)(l5)


In [ ]:
model_kaggle = tf.keras.Model(input_layer, outputs, name="model")

In [ ]:
checkpoint_filepath = './CHECKPOINTS'
model_checkpoint_callback = tf.keras.callbacks.ModelCheckpoint(
    filepath=checkpoint_filepath,
    save_weights_only=False,
    monitor='val_accuracy',
    mode='max',
    save_best_only=True)

checkpoint_filepath = './CHECKPOINTSloss'
model_checkpoint_callback2 = tf.keras.callbacks.ModelCheckpoint(
    filepath=checkpoint_filepath,
    save_weights_only=False,
    monitor='val_loss',
    mode='min',
    save_best_only=True)


In [ ]:
model_kaggle.compile(
    optimizer=tf.keras.optimizers.Adam(decay=3e-5),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=['accuracy'])

In [ ]:
model_kaggle.summary()


In [ ]:
model_kagglehist = model_kaggle.fit(ds, validation_data=val_dataset, epochs=400,
                                    callbacks=[PlotLossesKeras(),
                                               model_checkpoint_callback,
                                               model_checkpoint_callback2],
                                    verbose=False, shuffle=True)

In [ ]:
dftest = load_data('data_test')


In [ ]:
arrtest=dftest['images']

In [ ]:

arrtest = np.uint8(arrtest) / 255.
arrtest = defective_pixels(arrtest)
arrtest[0]

In [ ]:
plt.imshow(arrtest[6])

In [ ]:
arrtest.shape



In [ ]:
predictions = model_kaggle.predict(arrtest)

In [ ]:
classes = np.argmax(predictions, axis=1)
classes
# 5 4 1 ... 0 9 2

In [ ]:
plt.figure(figsize=(15,15))
for i, j in zip(range(60, 100), range(9)):
    plt.subplot(3,3,j+1)
    plt.xticks([])
    plt.yticks([])
    plt.grid(False)
    plt.imshow(np.array(arrtest[i]))
    plt.xlabel(classes[i])
plt.show()

In [ ]:
with open('13.csv', 'w') as f:
    f.write('Id,Category\n')
    for i, j in enumerate(classes):
        f.write(f'{i},{j}\n')

In [ ]:
f1 = 'submissionadam4.csv'
f2 = 'submissionadam5.csv'
for i, j in zip(open(f1, 'r').readlines(), open(f2, 'r').readlines()):
    if i != j:
        print(i, j)

In [ ]:

plt.imshow(np.array(arrtest[137]))


In [ ]:
mod = tf.keras.models.load_model('CHECKPOINTS/')

In [ ]:
with open('15bestacc.csv', 'w') as f:
    f.write('Id,Category\n')
    for i, j in enumerate(classes):
        f.write(f'{i},{j}\n')